[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_00/MFM_ch0_bvb_history/MFM_ch0_bvb_history.ipynb)

# MFM_ch0_bvb_history

**Modelling Financial Markets (MFM) — Chapter 0: Financial Markets in 2026**

The BET index of the Bucharest Stock Exchange, official closing values from its launch on 19 September 1997 to September 2026, on a log scale, with the 1999 low, the 2007 peak, the 2009 trough, the December 2018 bank-tax announcement, COVID-19 and the 2023 Hidroelectrica IPO.

*Author: Daniel Traian Pele*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_0'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import io
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

## Style and data loader

In [ ]:
# Stil standard MFM (identic cu SFM): transparent + ENG + legenda jos
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Culori brand
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Gray     = '#7F7F7F'
LightGray = '#DADADA'
Teal     = '#17A2B8'


def save_fig(name):
    """Salveaza figura ca PDF si PNG transparent, in folderul curent."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Plaseaza legenda in afara graficului, jos-centru."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def log_axis(ax, ticks):
    """Axa logaritmica cu etichete lizibile (fara notatie stiintifica)."""
    ax.set_yscale('log')
    ax.set_yticks(ticks)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:g}'))
    ax.yaxis.set_minor_formatter(plt.NullFormatter())


In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
LOCAL_DIRS = ['../../../data/market', '../../data/market', '../data/market', 'data/market']   # repo clonat
START, END = '2000-01-01', '2026-09-18'

# nume -> (sursa, simbol, camp)
SERIES = {
    'S&P 500': ('market', 'GSPC.INDX', 'close'),
    'Euro Stoxx 50': ('market', 'STOXX50E.INDX', 'close'),
    'Nikkei 225': ('market', 'N225.INDX', 'close'),
    'VIX': ('market', 'VIX.INDX', 'close'),
    'BET-TR': ('market', 'BETTR.INDX', 'close'),
    'BET': ('market', 'BET', 'close'),
    'BET-XT': ('market', 'BETXT.INDX', 'close'),
    'US Treasuries 20y+ (TLT)': ('market', 'TLT.US', 'adjusted_close'),
    'SPY': ('market', 'SPY.US', 'adjusted_close'),
    'RSP': ('market', 'RSP.US', 'adjusted_close'),
    'IBIT': ('market', 'IBIT.US', 'close'),
    'IBIT volume': ('market', 'IBIT.US', 'volume'),
    'Gold': ('market', 'XAUUSD.FOREX', 'close'),
    'EUR/USD': ('market', 'EURUSD.FOREX', 'close'),
    'EUR/RON (market file)': ('market', 'EURRON.FOREX', 'close'),
    'Bitcoin': ('market', 'BTC-USD.CC', 'close'),
    'Ethereum': ('market', 'ETH-USD.CC', 'close'),
    'Romania 10y': ('market', 'RO10Y.GBOND', 'close'),
    'Germany 10y': ('market', 'DE10Y.GBOND', 'close'),
    'Banca Transilvania': ('market', 'TLV.RO', 'adjusted_close'),
    'OMV Petrom': ('market', 'SNP.RO', 'adjusted_close'),
    'BRD': ('market', 'BRD.RO', 'adjusted_close'),
    'Transgaz': ('market', 'TGN.RO', 'adjusted_close'),
    'Romgaz': ('market', 'SNG.RO', 'adjusted_close'),
    'Hidroelectrica': ('market', 'H2O.RO', 'adjusted_close'),
    'EUR/RON': ('bnr', 'EUR', None),
    'DGS10': ('fred', 'DGS10', None),
    'DGS2': ('fred', 'DGS2', None),
    'FEDFUNDS': ('fred', 'FEDFUNDS', None),
    'USD per EUR': ('fred', 'DEXUSEU', None),
    'JPY per USD': ('fred', 'DEXJPUS', None),
    'T10Y2Y': ('fred', 'T10Y2Y', None),
    'Stablecoins': ('defillama', 'stablecoincharts/all', None),
}


def read_market(symbol):
    """Fisierul de date de piata: local (repo clonat), altfel din repo-ul GitHub."""
    for d in LOCAL_DIRS:
        path = os.path.join(d, f'{symbol}.csv')
        if os.path.exists(path):
            return pd.read_csv(path, index_col='date', parse_dates=True)
    return pd.read_csv(REPO_RAW + f'{symbol}.csv', index_col='date', parse_dates=True)


def fetch_daily(symbol, source='market', field='close', start=START, end=END):
    """Punctul unic de acces la date: intoarce o pd.Series zilnica (fara NaN)."""
    if source == 'market':
        s = read_market(symbol)[field]
    elif source == 'fred':
        raw = urllib.request.urlopen(
            f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={symbol}', timeout=90).read().decode()
        s = pd.read_csv(io.StringIO(raw), index_col=0, parse_dates=True, na_values='.').iloc[:, 0]
    elif source == 'defillama':
        req = urllib.request.Request(f'https://stablecoins.llama.fi/{symbol}',
                                     headers={'User-Agent': 'Mozilla/5.0'})
        d = json.load(urllib.request.urlopen(req, timeout=90))
        s = pd.Series({pd.to_datetime(int(x['date']), unit='s'): x['totalCirculatingUSD'].get('peggedUSD', 0)
                       for x in d}) / 1e9                                   # miliarde USD
    elif source == 'bnr':
        # cursul oficial BNR (RON pentru o unitate de valuta), arhive XML anuale
        import re
        rows = []
        for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
            url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
            xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                         timeout=90).read().decode()
            for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
                m = re.search(rf'<Rate currency="{symbol}">([\d.]+)</Rate>', body)
                if m:
                    rows.append((d, float(m.group(1))))
        s = pd.Series(dict(rows))
    else:
        raise ValueError(f'unknown source: {source}')
    s = pd.to_numeric(s, errors='coerce')
    s.index = pd.to_datetime(s.index)
    s.index.name = 'Date'
    return s.sort_index().loc[start:end].dropna()


def load(name, start=START, end=END):
    """Seria cu numele din SERIES (pentru aur si EUR/USD: doar zilele lucratoare)."""
    source, symbol, field = SERIES[name]
    s = fetch_daily(symbol, source, field, start, end).rename(name)
    if name in WEEKDAYS_ONLY:
        s = s[s.index.dayofweek < 5]
    return s


def load_panel(names, start=START, end=END):
    """Mai multe serii aliniate pe data (NaN acolo unde o piata nu tranzactioneaza)."""
    return pd.concat([load(n, start, end) for n in names], axis=1)


## Data

## Charts

In [ ]:
def fig_bvb_history():
    """Indicele BET 1997-2026 pe scara log, valori oficiale de inchidere, cu evenimente marcate."""
    bet = load('BET', start='1997-01-01').dropna()
    full = bet
    fig, ax = plt.subplots(figsize=(7.2, 3.3))
    ax.plot(bet.index, bet.values, color=MainBlue, lw=0.8, label='BET, official closing values (base 1,000 on 19 Sep 1997)')
    ax.set_yscale('log')
    ax.set_ylabel('Index level (log scale)')
    events = [('1999-04-21', 'Apr 1999\nlow', 0, 18), ('2007-07-24', 'Jul 2007\npeak', 0, 14),
              ('2009-02-25', 'Feb 2009\ntrough', 0, -26), ('2018-12-18', 'Dec 2018\nbank-tax\nannouncement', -34, -16),
              ('2020-03-16', 'Mar 2020\nCOVID-19', 16, -16), ('2023-07-12', 'Jul 2023\nHidroelectrica\nIPO', 0, -16)]
    for d, lab, dx, dy in events:
        t = full.index[full.index.searchsorted(pd.Timestamp(d))]
        ax.plot(t, full.loc[t], 'o', ms=3.5, color=IDAred)
        ax.annotate(lab, xy=(t, full.loc[t]), xytext=(dx, dy), textcoords='offset points', ha='center',
                    va='bottom' if dy > 0 else 'top', fontsize=6.5, color='black',
                    arrowprops=dict(arrowstyle='-', color=LightGray, lw=0.5))
    ax.set_ylim(150, 5e4)
    ax.set_title('The BET index, 1997-2026', fontsize=9, loc='left')
    legend_outside_bottom(ax, ncol=1, y=-0.13)
    plt.tight_layout()
    save_fig('ch0_bvb_history')
    lvl = lambda s, d: s.loc[s.index[s.index.searchsorted(pd.Timestamp(d))]]
    return dict(bet_first=(bet.index[0].date(), bet.iloc[0]),                 bet_1999_min=(bet.loc['1999'].idxmin().date(), bet.loc['1999'].min()),
                bet_2007_max=(bet.loc[:'2008'].idxmax().date(), bet.loc[:'2008'].max()),
                bet_2009_min=(bet.loc['2008-06':'2009-12'].idxmin().date(), bet.loc['2008-06':'2009-12'].min()),
                dec2018=(lvl(bet, '2018-12-18'), lvl(bet, '2018-12-19'), lvl(bet, '2018-12-19') / lvl(bet, '2018-12-18') - 1),
                bet_last=(bet.index[-1].date(), bet.iloc[-1]), bet_max=(bet.idxmax().date(), bet.max()),
                multiple_1997_2026=bet.iloc[-1] / bet.iloc[0])

## Run

In [ ]:
print(fig_bvb_history())

![ch0_bvb_history](./ch0_bvb_history.png)

Output: `ch0_bvb_history.pdf`